# Pakistani Clothing Brands — Big Data Analytics Pipeline
### Clean, organized Google Colab notebook | YouTube API → One CSV → Topic Clustering → Sentiment → Verified Product Clustering

**Run from top to bottom.** Collection is real-data API based; **million records are a capacity target, not guaranteed**. The first part produces `youtube_clothing_ONE_FILE.csv`; subsequent parts create enriched and clustered outputs. **Verified prices require an authorized product catalog; commenter age/gender/city cannot be obtained from YouTube comment API.**

**Suggested approach:** Set `VIDEOS_PER_BRAND=5` and `MAX_COMMENTS_PER_VIDEO=100` for an initial check; scale after confirming API and quota. Re-running collection resumes from SQLite. Never commit your API key.

**Colab runtime:** Runtime → Change runtime type → GPU recommended for multilingual sentiment inference.


## 1 · Install compatible libraries (run once, then restart runtime if imports fail)

In [4]:
%pip uninstall -y gradio gradio-client diffusers timm torchvision

%pip install -q \
    "huggingface-hub==0.36.2" \
    "transformers==4.57.6" \
    "datasets==4.8.5" \
    "bottleneck>=1.5.0"

%pip check

fastai 2.8.12 requires torchvision, which is not installed.
hf-gradio 0.4.1 requires gradio-client, which is not installed.
ipython 7.34.0 requires jedi, which is not installed.


## 2 · Connect Google Drive, enter YouTube API key, set collection limits

In [5]:
import os, re, time, json, sqlite3, csv
from pathlib import Path
from getpass import getpass
from datetime import datetime, timezone
from urllib.parse import urlparse
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from tqdm.auto import tqdm
import pandas as pd

USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/clothing_big_data')
else:
    OUT = Path('/content/clothing_big_data')
OUT.mkdir(parents=True, exist_ok=True)
DB_FILE = OUT / 'youtube_collection.sqlite3'
CSV_FILE = OUT / 'youtube_clothing_ONE_FILE.csv'

API_KEY = getpass('Paste ONLY your YouTube Data API key: ').strip()
if not API_KEY.startswith('AIza'):
    raise ValueError('Google API key should begin with AIza. Do not paste Python code here.')
youtube = build('youtube', 'v3', developerKey=API_KEY, cache_discovery=False)

BRANDS = ['Khaadi','Sapphire','Gul Ahmed','Alkaram Studio','Nishat Linen','Limelight','Beechtree','Bonanza Satrangi','J.','Ethnic']
VIDEOS_PER_BRAND = 100  # Increase gradually; each search page consumes quota
MAX_COMMENTS_PER_VIDEO = 3000  # Actual accessible comments may be much lower
PAUSE_SECONDS = 0.15
print('Database:', DB_FILE)
print('Output CSV:', CSV_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Paste ONLY your YouTube Data API key: ··········
Database: /content/drive/MyDrive/clothing_big_data/youtube_collection.sqlite3
Output CSV: /content/drive/MyDrive/clothing_big_data/youtube_clothing_ONE_FILE.csv


## 3 · Test YouTube connection before a long run

In [6]:
try:
    r = youtube.videos().list(part='snippet', id='dQw4w9WgXcQ').execute()
    print('API connection succeeded. Video records:', len(r.get('items', [])))
except HttpError as e:
    raise RuntimeError(f'API request failed (HTTP {e.resp.status}): {e}')

API connection succeeded. Video records: 1


## 4 · Extract candidate product categories, URLs and price mentions

In [7]:
CATEGORY_KEYWORDS = {
    '3 Piece Suit': ['3 piece','3-piece','3pc','3 pcs'],
    '2 Piece Suit': ['2 piece','2-piece','2pc','2 pcs'],
    'Kurti': ['kurti','kurta'], 'Lawn': ['lawn'],
    'Shirt': ['shirt'], 'Abaya': ['abaya'], 'Dress': ['dress','frock']
}
PRICE_RE = re.compile(r'(?<!\w)(?:PKR|Rs\.?|₨)\s*([0-9][0-9,]*(?:\.[0-9]{1,2})?)\b|\b([0-9][0-9,]*(?:\.[0-9]{1,2})?)\s*(?:PKR|Rs\.?)\b', re.I)
URL_RE = re.compile(r'https?://[^\s<>"\']+',re.I)

def product_candidate(title, desc):
    combined = f'{title or ""}\n{desc or ""}'
    low = combined.lower()
    cat = next((k for k,v in CATEGORY_KEYWORDS.items() if any(x in low for x in v)), 'Unknown')
    vals = []
    for x,y in PRICE_RE.findall(combined):
        try: vals.append(float((x or y).replace(',','')))
        except ValueError: pass
    links = [s.rstrip('.,);') for s in URL_RE.findall(desc or '')]
    return dict(product_candidate=title,product_category_candidate=cat,
                price_mention_pkr=vals[0] if len(vals)==1 else None,
                price_mentions_count=len(vals), price_source='video_text_unverified' if vals else 'not_found',
                verified_price_pkr=None, product_match_status='unverified',
                product_url_candidate=next((u for u in links if any(z in urlparse(u).netloc.lower() for z in ['khaadi','sapphire','gulahmed','alkaram','daraz'])), None))
print(product_candidate('3 Piece Suit Rs. 4,500','Shop link https://example.com'))

{'product_candidate': '3 Piece Suit Rs. 4,500', 'product_category_candidate': '3 Piece Suit', 'price_mention_pkr': 4500.0, 'price_mentions_count': 1, 'price_source': 'video_text_unverified', 'verified_price_pkr': None, 'product_match_status': 'unverified', 'product_url_candidate': None}


## 5 · Initialize persistent SQLite storage

In [8]:
con = sqlite3.connect(str(DB_FILE), timeout=60)
con.execute('PRAGMA busy_timeout=60000')
con.execute('CREATE TABLE IF NOT EXISTS videos (video_id TEXT PRIMARY KEY, data_json TEXT NOT NULL)')
con.execute('CREATE TABLE IF NOT EXISTS comments (comment_id TEXT PRIMARY KEY, video_id TEXT NOT NULL, data_json TEXT NOT NULL)')
con.execute('CREATE INDEX IF NOT EXISTS idx_comments_video ON comments(video_id)')
con.execute('CREATE TABLE IF NOT EXISTS completed (video_id TEXT PRIMARY KEY, completed_at TEXT NOT NULL)')
con.commit()
print('Previously saved comments:', con.execute('SELECT COUNT(*) FROM comments').fetchone()[0])

Previously saved comments: 23647


## 6 · Discover brand videos and save video metadata

In [9]:
def search_ids(brand, limit):
    ids=[]; seen=set(); used_pages=set(); token=None
    while len(ids) < limit:
        args = dict(part='snippet',q=f'{brand} Pakistan clothing collection',type='video',maxResults=min(50,limit-len(ids)),order='relevance')
        if token: args['pageToken']=token
        data = youtube.search().list(**args).execute()
        items = data.get('items',[])
        if not items: break
        for item in items:
            vid=item.get('id',{}).get('videoId')
            if vid and vid not in seen: ids.append(vid); seen.add(vid)
        nxt=data.get('nextPageToken')
        if not nxt or nxt in used_pages: break
        used_pages.add(nxt);token=nxt
        time.sleep(PAUSE_SECONDS)
    return ids[:limit]

def store_video_details(ids,brand):
    added=0
    for offset in range(0,len(ids),50):
        r=youtube.videos().list(part='snippet,statistics',id=','.join(ids[offset:offset+50])).execute()
        for item in r.get('items',[]):
            sn=item.get('snippet',{});st=item.get('statistics',{});vid=item['id']
            title=sn.get('title','');desc=sn.get('description','')
            row=dict(platform='YouTube',brand_search_term=brand,video_id=vid,
                     video_url=f'https://www.youtube.com/watch?v={vid}',video_title=title,
                     video_description=desc,channel_title=sn.get('channelTitle'),channel_id=sn.get('channelId'),
                     video_published_at=sn.get('publishedAt'),video_views=int(st['viewCount']) if 'viewCount' in st else None,
                     video_likes=int(st['likeCount']) if 'likeCount' in st else None,
                     reported_comment_count=int(st['commentCount']) if 'commentCount' in st else None,
                     **product_candidate(title,desc))
            con.execute('INSERT OR IGNORE INTO videos(video_id,data_json) VALUES (?,?)',(vid,json.dumps(row,ensure_ascii=False)))
            added+=1
        con.commit();time.sleep(PAUSE_SECONDS)
    return added

for brand in tqdm(BRANDS,desc='Searching brands'):
    try:
        ids=search_ids(brand,VIDEOS_PER_BRAND)
        count=store_video_details(ids,brand)
        print(f'{brand}: {count} video metadata records retrieved')
    except HttpError as e:
        print(f'HTTP {e.resp.status} on {brand}: {str(e)[:220]}')
        if e.resp.status in (403,429):
            print('Stopping on access/quota error. Re-run in a permitted later session.')
            break
print('Unique stored videos:',con.execute('SELECT COUNT(*) FROM videos').fetchone()[0])

Searching brands:   0%|          | 0/10 [00:00<?, ?it/s]

Khaadi: 100 video metadata records retrieved
Sapphire: 100 video metadata records retrieved
Gul Ahmed: 100 video metadata records retrieved
Alkaram Studio: 100 video metadata records retrieved
Nishat Linen: 100 video metadata records retrieved
Limelight: 100 video metadata records retrieved
HTTP 429 on Beechtree: <HttpError 429 when requesting https://youtube.googleapis.com/youtube/v3/search?part=snippet&q=Beechtree+Pakistan+clothing+collection&type=video&maxResults=8&order=relevance&pageToken=CGQQAA&key=AIzaSyAMwbjgqvnXoKTU75U7e
Stopping on access/quota error. Re-run in a permitted later session.
Unique stored videos: 1150


## 7 · Collect top-level comments with pagination and resume support

In [10]:
def collect_comments_for_video(video_id,limit=3000):
    if con.execute('SELECT 1 FROM completed WHERE video_id=?',(video_id,)).fetchone():
        return 0
    token=None; received=0; fresh=0
    while received < limit:
        args=dict(part='snippet',videoId=video_id,maxResults=min(100,limit-received),textFormat='plainText',order='time')
        if token:args['pageToken']=token
        try:
            data=youtube.commentThreads().list(**args).execute()
        except HttpError as e:
            # Comments disabled or inaccessible: skip; do not mark as completed, so later attempts are possible
            if e.resp.status in (403,404) and any(x in str(e).lower() for x in ['commentsdisabled','videonotfound','forbidden']):
                print('Unavailable comments:',video_id)
                return fresh
            raise
        items=data.get('items',[])
        for item in items:
            c=item.get('snippet',{}).get('topLevelComment',{})
            sn=c.get('snippet',{}); message=sn.get('textDisplay','').strip()
            if not message:continue
            row=dict(comment_id=c.get('id'),video_id=video_id,comment_text=message,
                     comment_likes=sn.get('likeCount',0),comment_published_at=sn.get('publishedAt'),
                     comment_updated_at=sn.get('updatedAt'))
            cur=con.execute('INSERT OR IGNORE INTO comments(comment_id,video_id,data_json) VALUES (?,?,?)',
                            (row['comment_id'],video_id,json.dumps(row,ensure_ascii=False)))
            fresh+=cur.rowcount
        con.commit()
        received+=len(items)
        token=data.get('nextPageToken')
        if not token or not items:break
        time.sleep(PAUSE_SECONDS)
    con.execute('INSERT OR REPLACE INTO completed VALUES (?,?)', (video_id,datetime.now(timezone.utc).isoformat()))
    con.commit()
    return fresh

ids=[r[0] for r in con.execute('SELECT video_id FROM videos ORDER BY video_id')]
for vid in tqdm(ids,desc='Collecting comments'):
    try:
        collect_comments_for_video(vid,MAX_COMMENTS_PER_VIDEO)
    except HttpError as e:
        print('Collection interrupted at video:',vid,'HTTP:',e.resp.status)
        print('Progress saved. Fix the access/quota error then rerun Cell 7.')
        break
print('Unique saved comments:',con.execute('SELECT COUNT(*) FROM comments').fetchone()[0])
print('Completed videos:',con.execute('SELECT COUNT(*) FROM completed').fetchone()[0])

Unavailable comments: 11b0Bch7ZFI


Unavailable comments: 21CP4sxj6cE


Unavailable comments: GX-MFg20HqQ


Unavailable comments: MW8UFRbj-PU


Unavailable comments: OBV38yM-oXM


Unavailable comments: UHBnNotHhU4


Unavailable comments: a2LTTjoiw1k


Unavailable comments: fy9KbOwoanA


Unavailable comments: js61nTJghXk


Unavailable comments: xrdUCfCXSNI
Unique saved comments: 24407
Completed videos: 1140


## 8 · Export all available comments to ONE CSV

In [11]:
COLUMNS = [
 'platform','brand_search_term','video_id','video_url','video_title','channel_title','channel_id','video_published_at',
 'video_views','video_likes','reported_comment_count',
 'product_candidate','product_category_candidate','product_url_candidate','price_mention_pkr','price_mentions_count',
 'price_source','verified_price_pkr','product_match_status',
 'comment_id','comment_text','comment_likes','comment_published_at','comment_updated_at',
 'sentiment','price_complaint','collected_at'
]

def export_single_csv():
    count=0
    with open(CSV_FILE,'w',newline='',encoding='utf-8-sig') as f:
        w=csv.DictWriter(f,fieldnames=COLUMNS,extrasaction='ignore')
        w.writeheader()
        cur=con.execute('SELECT c.data_json,v.data_json FROM comments c JOIN videos v ON v.video_id=c.video_id ORDER BY c.comment_id')
        now=datetime.now(timezone.utc).isoformat()
        for comment_json,video_json in cur:
            item={**json.loads(video_json),**json.loads(comment_json)}
            item.update(sentiment='',price_complaint='',collected_at=now)
            w.writerow(item)
            count+=1
    return count

rows=export_single_csv()
print('Exported rows:',rows)
print('Single CSV:',CSV_FILE)
print('File size MB:',round(CSV_FILE.stat().st_size/1024**2,2))

Exported rows: 24407
Single CSV: /content/drive/MyDrive/clothing_big_data/youtube_clothing_ONE_FILE.csv
File size MB: 11.36


## 9 · Check collected record count (safe for large files)

In [12]:
if CSV_FILE.exists() and CSV_FILE.stat().st_size:
    print('CSV:', CSV_FILE)
    print('Size (MB):', round(CSV_FILE.stat().st_size/1024**2, 2))
    print('Sample:')
    display(pd.read_csv(CSV_FILE, nrows=5)[['brand_search_term','video_title','comment_text','product_category_candidate','price_mention_pkr']])
    total_rows = sum(len(c) for c in pd.read_csv(CSV_FILE, usecols=['comment_id'], chunksize=50_000))
    print(f'Actual saved comment rows: {total_rows:,}')
else:
    print('No collection CSV yet; run Sections 2–8 first.')


CSV: /content/drive/MyDrive/clothing_big_data/youtube_clothing_ONE_FILE.csv
Size (MB): 11.36
Sample:


,brand_search_term,video_title,comment_text,product_category_candidate,price_mention_pkr
0,Gul Ahmed,Gul Ahmed Two Piece Collection Summer Vol 3 2016,very surprb design,Kurti,NaN
1,Gul Ahmed,Gul Ahmed Latest Winter Amazing Collection 2016,ok,Kurti,NaN
2,Alkaram Studio,Alkaram Studio Behind the scenes Allure collec...,Alkaram Studio Behind the scenes Allure collec...,3 Piece Suit,NaN
3,Gul Ahmed,Gul Ahmed Spring-Summer 2016 Premium Embroider...,Very nice dress,Kurti,NaN
4,Gul Ahmed,Gul Ahmed Latest Winter Collection 2016,maien suite lyna chati hi koi nub,Kurti,NaN


Actual saved comment rows: 24,407


---
# Part B · Comment topic clustering (unsupervised)
**Features:** TF-IDF word/bi-gram representations of comments; *not* positive/negative sentiment. MiniBatchKMeans clusters comments by word-use similarity. Outputs will be stored under `clustering_results/`.


In [13]:
import numpy as np, re, json, joblib
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

CSV_PATH = CSV_FILE
RESULTS = OUT / 'clustering_results'
RESULTS.mkdir(parents=True, exist_ok=True)
CHUNK_SIZE = 50_000
assert CSV_PATH.exists(), f'Run collection/export first: {CSV_PATH}'
header = pd.read_csv(CSV_PATH,nrows=0).columns.tolist()
assert {'comment_text','video_id','brand_search_term'}.issubset(header)
row_count = sum(len(c) for c in pd.read_csv(CSV_PATH,usecols=['comment_text'],chunksize=CHUNK_SIZE))
print(f'Comments available for topic clustering: {row_count:,}')
assert row_count > 0, 'CSV empty: collect comments first.'


Comments available for topic clustering: 24,407


## 10 · Sample texts for TF-IDF training

In [14]:
import random
RANDOM_SEED = 42
SAMPLE_SIZE = min(50_000, row_count)
rng = random.Random(RANDOM_SEED)
reservoir = []
seen = 0
for chunk in tqdm(pd.read_csv(CSV_PATH, usecols=['comment_text'], chunksize=CHUNK_SIZE), desc='Sampling comments'):
    for val in chunk['comment_text'].dropna().astype(str):
        text = re.sub(r'\s+', ' ', val).strip()
        if not text or len(text)<3:
            continue
        seen += 1
        if len(reservoir) < SAMPLE_SIZE:
            reservoir.append(text)
        else:
            j = rng.randrange(seen)
            if j < SAMPLE_SIZE:
                reservoir[j] = text
print('Eligible text rows:', seen, 'Training sample:', len(reservoir))
assert len(reservoir) >= 30, 'Not enough nonempty comments to cluster.'

Sampling comments: 0it [00:00, ?it/s]

Eligible text rows: 23868 Training sample: 23868


## 11 · Train and evaluate MiniBatch K-Means topic clusters

In [15]:
vectorizer = TfidfVectorizer(
    lowercase=True, strip_accents='unicode',
    ngram_range=(1,2), min_df=2, max_df=0.95,
    max_features=25_000, sublinear_tf=True,
    token_pattern=r'(?u)\b\w+\b'
)
X = vectorizer.fit_transform(reservoir)
print('Sparse TF-IDF shape:', X.shape)

# Choose cluster count on a small sample to bound compute/time
K_VALUES = [k for k in [4,6,8] if k < len(reservoir)]
score_idx = np.random.default_rng(42).choice(len(reservoir), size=min(3000, len(reservoir)), replace=False)
scores = {}
for k in K_VALUES:
    model = MiniBatchKMeans(n_clusters=k, batch_size=2048, random_state=42, n_init=3)
    labels = model.fit_predict(X)
    if len(np.unique(labels[score_idx]))>1:
        scores[k] = float(silhouette_score(X[score_idx], labels[score_idx], metric='cosine'))
print('Sample silhouette scores:', scores)
assert scores, 'Could not evaluate topic clusters.'
best_k = max(scores, key=scores.get)
cluster_model = MiniBatchKMeans(n_clusters=best_k, batch_size=2048, random_state=42, n_init=5)
cluster_model.fit(X)
print('Selected topic cluster count:', best_k)

Sparse TF-IDF shape: (23868, 25000)
Sample silhouette scores: {4: 0.012775693725559324, 6: 0.018028079725561152, 8: 0.006677065163548683}
Selected topic cluster count: 6


## 12 · Inspect cluster keywords and save model

In [16]:
terms = vectorizer.get_feature_names_out()
centers = cluster_model.cluster_centers_
topic_words = {}
for cluster_id, center in enumerate(centers):
    words = [str(terms[i]) for i in center.argsort()[-12:][::-1]]
    topic_words[int(cluster_id)] = words
    print(f'Topic {cluster_id}:', ', '.join(words))

joblib.dump({'vectorizer':vectorizer, 'model':cluster_model}, RESULTS/'comment_topic_model.joblib')
(RESULTS/'topic_keywords.json').write_text(json.dumps(topic_words, indent=2), encoding='utf-8')

Topic 0: nice, i, the, is, to, a, it, price, fake, you, this, beautiful
Topic 1: who buys, all time, a granny, outdated, buys this, granny, open all, colours and, buys, to leave, and u, this so
Topic 2: famous, inki, me, here so, here to, hermes, hermes bags, hermes boutique, hero, hers, hers like, herself
Topic 3: لان, لان برانڈ, برانڈ, ایمبریڈری لان, دوپٹہ لان, اتحاد, ایمبریڈری, دوپٹہ, کولیکشن, پرینٹڈ, ھوم ڈلیوری, ھوم
Topic 4: dupata, missing, oder, hota hai, karain, hota, online, hai, to, hi, hey, hes
Topic 5: we love, scene, no hate, indians, hate, love you, wala, we, love, no, hai, you


1462

## 13 · Add topic cluster to entire CSV in chunks

In [17]:
ANNOTATED_CSV = RESULTS/'youtube_clothing_clustered_ONE_FILE.csv'
first = True
for chunk in tqdm(pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE, low_memory=False), desc='Predicting all comments'):
    texts = chunk['comment_text'].fillna('').astype(str)
    transformed = vectorizer.transform(texts)
    chunk['comment_topic_cluster'] = cluster_model.predict(transformed).astype('int16')
    # Blank strings: unknown topic, not a real language cluster
    chunk.loc[texts.str.strip().eq(''), 'comment_topic_cluster'] = -1
    chunk.to_csv(ANNOTATED_CSV, mode='w' if first else 'a', header=first, index=False, encoding='utf-8-sig' if first else 'utf-8')
    first = False
print('Combined clustered CSV:', ANNOTATED_CSV)

Predicting all comments: 0it [00:00, ?it/s]

Combined clustered CSV: /content/drive/MyDrive/clothing_big_data/clustering_results/youtube_clothing_clustered_ONE_FILE.csv


## 14 · Cluster YouTube videos by engagement and topic shares

In [18]:
topic_counts = Counter()
video_stats = {}
for chunk in tqdm(pd.read_csv(ANNOTATED_CSV, chunksize=CHUNK_SIZE, low_memory=False), desc='Aggregating videos'):
    chunk['comment_topic_cluster'] = pd.to_numeric(chunk['comment_topic_cluster'],errors='coerce').fillna(-1).astype(int)
    counts = chunk.groupby(['video_id','comment_topic_cluster']).size()
    for (vid,topic), count in counts.items():
        topic_counts[(vid,int(topic))] += int(count)
    # Each video's metadata is repeated across comment rows. Store once.
    selected = ['video_id','brand_search_term']
    for col in ['video_views','video_likes','product_category_candidate','verified_price_pkr','product_match_status']:
        if col in chunk.columns: selected.append(col)
    for row in chunk[selected].drop_duplicates('video_id').to_dict('records'):
        video_stats.setdefault(row['video_id'],row)

rows = []
for vid, metadata in video_stats.items():
    row = dict(metadata)
    topic_total = sum(v for (v_id,t),v in topic_counts.items() if v_id == vid and t>=0)
    row['collected_comments'] = topic_total
    for k in range(best_k):
        row[f'topic_{k}_share'] = topic_counts.get((vid,k),0)/topic_total if topic_total else 0
    rows.append(row)
videos = pd.DataFrame(rows)
print('Videos with collected comments:',len(videos))

for col in ['video_views','video_likes']:
    if col not in videos: videos[col] = np.nan
    videos[col] = pd.to_numeric(videos[col],errors='coerce')
videos['like_rate'] = (videos['video_likes'] / videos['video_views'].replace(0,np.nan)).clip(0,1)
videos['log_views'] = np.log1p(videos['video_views'].clip(lower=0))
videos['log_comment_count'] = np.log1p(videos['collected_comments'])
video_features = ['log_views','log_comment_count','like_rate'] + [f'topic_{k}_share' for k in range(best_k)]
valid = videos.dropna(subset=video_features).copy()
if len(valid) >= 6:
    scaler = StandardScaler()
    Z = scaler.fit_transform(valid[video_features])
    k_video = min(4, max(2,len(valid)//3))
    video_model = MiniBatchKMeans(n_clusters=k_video,random_state=42,n_init=5,batch_size=1024)
    valid['video_cluster'] = video_model.fit_predict(Z)
    videos = videos.merge(valid[['video_id','video_cluster']],on='video_id',how='left')
    joblib.dump({'scaler':scaler,'model':video_model,'features':video_features},RESULTS/'video_cluster_model.joblib')
else:
    videos['video_cluster'] = pd.NA
    print('Not enough complete video records for video clustering.')
videos.to_csv(RESULTS/'video_clusters.csv',index=False)
print('Saved video clusters:',RESULTS/'video_clusters.csv')
print(videos[['brand_search_term','video_id','collected_comments','video_cluster']].head())

Aggregating videos: 0it [00:00, ?it/s]

Videos with collected comments: 761
Saved video clusters: /content/drive/MyDrive/clothing_big_data/clustering_results/video_clusters.csv
  brand_search_term     video_id  collected_comments  video_cluster
0         Gul Ahmed  MPyA6jQCQrw                   2            3.0
1         Gul Ahmed  p7ze2rcV0B8                   3            3.0
2    Alkaram Studio  NNHJQnjB5do                   2            2.0
3         Gul Ahmed  1mMS76QIV9A                   2            3.0
4         Gul Ahmed  uPlYDevpvSs                   4            3.0


---
# Part C · Multilingual sentiment + verified product enrichment
**Important:** The classifier makes *predictions*, not ground-truth labels; evaluate on manually labelled English/Roman Urdu/Urdu comments. Verified prices remain blank until you provide real product listings and evidence. Age, gender and city remain blank for YouTube commenters.

**If the earlier `torchvision::nms` import error persists:** rerun Section 1, then **Runtime → Restart session** and rerun from Section 2. `HF_TOKEN` warning for a public model can be ignored.


In [19]:
import csv, os, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
BASE = OUT
SOURCE = RESULTS/'youtube_clothing_clustered_ONE_FILE.csv'
if not SOURCE.exists():
    SOURCE = CSV_FILE
assert SOURCE.exists(), 'Run collection/export first.'
OUTPUT = BASE/'youtube_clothing_enriched_ONE_FILE.csv'
CATALOG = BASE/'verified_product_catalog.csv'
MATCHES = BASE/'verified_video_product_matches.csv'
BATCH_SIZE = 16 if torch.cuda.is_available() else 8
ENRICH_CHUNK_SIZE = 1000
MAX_ROWS = 1000   # First test 1000 rows; set to None for complete output.
print('Source:', SOURCE, 'GPU:',torch.cuda.is_available())


Source: /content/drive/MyDrive/clothing_big_data/clustering_results/youtube_clothing_clustered_ONE_FILE.csv GPU: True


## 15 · Create catalog templates (fill with genuine verified sources)

In [20]:
import pandas as pd
if not CATALOG.exists():
    pd.DataFrame(columns=['product_id','brand','product_name','product_category','verified_price_pkr','product_url','price_checked_at','verification_status']).to_csv(CATALOG,index=False)
if not MATCHES.exists():
    pd.DataFrame(columns=['video_id','product_id','match_evidence_url','match_status']).to_csv(MATCHES,index=False)
print('Fill these and rerun this cell before final export:')
print(CATALOG)
print(MATCHES)

Fill these and rerun this cell before final export:
/content/drive/MyDrive/clothing_big_data/verified_product_catalog.csv
/content/drive/MyDrive/clothing_big_data/verified_video_product_matches.csv


## 16 · Validate catalog and evidence-based video-product matches

In [21]:
import numpy as np
catalog=pd.read_csv(CATALOG,dtype={'product_id':str,'brand':str})
links=pd.read_csv(MATCHES,dtype={'video_id':str,'product_id':str})
required_cat=['product_id','verified_price_pkr','product_url','price_checked_at','verification_status']
required_links=['video_id','product_id','match_evidence_url','match_status']
assert set(required_cat).issubset(catalog.columns)
assert set(required_links).issubset(links.columns)
if not catalog.empty:
    catalog['verified_price_pkr']=pd.to_numeric(catalog['verified_price_pkr'],errors='coerce')
    catalog=catalog[(catalog.verification_status.astype(str).str.lower()=='verified') & catalog.verified_price_pkr.gt(0) & catalog.product_url.notna() & catalog.price_checked_at.notna()].copy()
    assert not catalog.product_id.duplicated().any(), 'Duplicate verified product IDs'
if not links.empty:
    links=links[links.match_status.astype(str).str.lower()=='verified'].copy()
    links=links.dropna(subset=['video_id','product_id','match_evidence_url'])
    links=links[links.product_id.isin(catalog.product_id)]
    # all multi-product or conflicting matches are excluded from comment-level assignment
    good_ids=links.groupby('video_id').product_id.nunique()
    good_ids=good_ids[good_ids==1].index
    links=links[links.video_id.isin(good_ids)].drop_duplicates('video_id')
verified=links.merge(catalog,on='product_id',how='inner',validate='many_to_one')
verified=verified.rename(columns={'verified_price_pkr':'catalog_verified_price_pkr','product_name':'catalog_product_name','product_category':'catalog_category','product_url':'catalog_product_url','price_checked_at':'catalog_price_checked_at'})
print('Verified distinct video-product mappings:',len(verified))

Verified distinct video-product mappings: 0


## 17 · Load multilingual sentiment classifier

In [22]:
import csv, os, torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
MODEL_ID='lxyuan/distilbert-base-multilingual-cased-sentiments-student'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
model=AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
device='cuda' if torch.cuda.is_available() else 'cpu'
model=model.to(device).eval()
print('Inference device:',device)
print('Labels:',model.config.id2label)

def predict_sentiment(texts):
    labels=[]; conf=[]
    for start in range(0,len(texts),BATCH_SIZE):
        batch=[str(x) if pd.notna(x) else '' for x in texts[start:start+BATCH_SIZE]]
        inputs=tokenizer(batch,return_tensors='pt',padding=True,truncation=True,max_length=128).to(device)
        with torch.inference_mode():
            probs=torch.softmax(model(**inputs).logits,dim=-1)
        scores,indices=probs.max(dim=-1)
        labels.extend([model.config.id2label[int(i)].capitalize() for i in indices.cpu()])
        conf.extend(scores.cpu().tolist())
    return labels,conf

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

Inference device: cuda
Labels: {0: 'positive', 1: 'neutral', 2: 'negative'}


## 18 · Stream sentiment predictions and catalog enrichment to a single CSV

In [23]:
PART=Path(str(OUTPUT)+('.test.part.csv' if MAX_ROWS is not None else '.part.csv'))
source_header=list(pd.read_csv(SOURCE,nrows=0).columns)
assert {'video_id','comment_text'}.issubset(source_header), 'Input must have video_id, comment_text'
new_cols=['sentiment_prediction','sentiment_confidence','product_id_verified','product_name_verified','product_category_verified','verified_price_pkr','verified_product_url','price_checked_at','price_verification_status','age_group','gender','city','demographic_source','price_complaint_keyword']
cols=list(dict.fromkeys(source_header+new_cols))

# Keep existing CSV sentiment as-is; prediction has a distinct column.
finished=0
if MAX_ROWS is None and PART.exists():
    with open(PART,encoding='utf-8-sig',newline='') as f:
        reader=csv.reader(f)
        part_header=next(reader,[])
        assert part_header==cols,'Partial output schema differs. Delete the .part.csv to restart.'
        finished=sum(1 for _ in reader)
print('Resuming after rows:',finished)

match_map={str(r.video_id):r for r in verified.itertuples(index=False)}
price_words=['expensive','overpriced','mehnga','mehngi','mehngay','price kam','bohat costly','too costly','zyada price']
mode='a' if MAX_ROWS is None and PART.exists() else 'w'
processed=0
with open(PART,mode,encoding='utf-8-sig' if mode=='w' else 'utf-8',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=cols,extrasaction='ignore')
    if mode=='w': writer.writeheader()
    for chunk in tqdm(pd.read_csv(SOURCE,chunksize=ENRICH_CHUNK_SIZE,low_memory=False,dtype={'video_id':str}),desc='Enriching'):
        initial=processed
        processed+=len(chunk)
        if MAX_ROWS is not None and initial>=MAX_ROWS:break
        if processed<=finished:continue
        if initial<finished:chunk=chunk.iloc[finished-initial:]
        if MAX_ROWS is not None:
            chunk=chunk.iloc[:max(0,MAX_ROWS-max(initial,finished))]
        if chunk.empty:continue
        labs,confidence=predict_sentiment(chunk.comment_text.fillna('').astype(str).tolist())
        for row,label,score in zip(chunk.to_dict('records'),labs,confidence):
            # normalize NaN to blank, preserve other original columns
            row={k:('' if pd.isna(v) else v) for k,v in row.items()}
            row['sentiment_prediction']=label
            row['sentiment_confidence']=round(float(score),5)
            hit=match_map.get(str(row.get('video_id','')))
            row['product_id_verified']=hit.product_id if hit is not None else ''
            row['product_name_verified']=hit.catalog_product_name if hit is not None else ''
            row['product_category_verified']=hit.catalog_category if hit is not None else ''
            row['verified_price_pkr']=hit.catalog_verified_price_pkr if hit is not None else ''
            row['verified_product_url']=hit.catalog_product_url if hit is not None else ''
            row['price_checked_at']=hit.catalog_price_checked_at if hit is not None else ''
            row['price_verification_status']='verified' if hit is not None else 'not_verified'
            row['age_group']=row['gender']=row['city']=''
            row['demographic_source']='unavailable_from_youtube_comments'
            row['price_complaint_keyword']=int(any(w in str(row['comment_text']).lower() for w in price_words))
            writer.writerow(row)
        f.flush()
# Only promote to final output once the full source was processed
if MAX_ROWS is None:
    os.replace(PART,OUTPUT)
    print('Complete export:',OUTPUT)
else:
    print('TEST MODE: partial file saved:',PART,'; set MAX_ROWS=None for full run.')

Resuming after rows: 0


Enriching: 0it [00:00, ?it/s]

TEST MODE: partial file saved: /content/drive/MyDrive/clothing_big_data/youtube_clothing_enriched_ONE_FILE.csv.test.part.csv ; set MAX_ROWS=None for full run.


## 19 · Product-level price/sentiment clustering (only verified product mappings)

In [24]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
if not OUTPUT.exists():
    print('Finish full sentiment export first (MAX_ROWS=None).')
else:
    agg={}
    for chunk in pd.read_csv(OUTPUT,chunksize=50000,low_memory=False):
        x=chunk[chunk['price_verification_status'].eq('verified') & chunk['product_id_verified'].notna()].copy()
        if x.empty:continue
        for pid,g in x.groupby('product_id_verified'):
            st=agg.setdefault(pid,{'n':0,'positive':0,'negative':0,'neutral':0,'price_complaints':0,'price':float(g['verified_price_pkr'].iloc[0]),'name':str(g['product_name_verified'].iloc[0])})
            st['n']+=len(g)
            counts=g['sentiment_prediction'].str.lower().value_counts()
            for key in ('positive','negative','neutral'):st[key]+=int(counts.get(key,0))
            st['price_complaints']+=int(pd.to_numeric(g['price_complaint_keyword'],errors='coerce').fillna(0).sum())
    features=pd.DataFrame([{'product_id':pid,'product_name':r['name'],'verified_price_pkr':r['price'],'comment_count':r['n'],'positive_share':r['positive']/r['n'],'negative_share':r['negative']/r['n'],'neutral_share':r['neutral']/r['n'],'price_complaint_share':r['price_complaints']/r['n']} for pid,r in agg.items()])
    if len(features)>=6:
        feat_cols=['verified_price_pkr','positive_share','negative_share','price_complaint_share']
        scaled=StandardScaler().fit_transform(features[feat_cols])
        choices={}
        for k in range(2,min(6,len(features)-1)+1):
            labels=KMeans(n_clusters=k,n_init=10,random_state=42).fit_predict(scaled)
            if len(set(labels))>=2:choices[k]=silhouette_score(scaled,labels)
        best=max(choices,key=choices.get)
        features['product_cluster']=KMeans(n_clusters=best,n_init=10,random_state=42).fit_predict(scaled)
        print('Optimal K:',best,'silhouette:',choices[best])
        display(features.groupby('product_cluster')[feat_cols].mean())
    else:
        features['product_cluster']=pd.NA
        print(f'Only {len(features)} verified products. Need >=6 for product clustering. Add real verified catalog matches.')
    features.to_csv(BASE/'verified_product_cluster_summary.csv',index=False)
    print('Product summary:',BASE/'verified_product_cluster_summary.csv')

Finish full sentiment export first (MAX_ROWS=None).


## 20 · Optional authorized demographic aggregate template

In [25]:
DEMOGRAPHICS=BASE/'authorized_demographic_aggregates_TEMPLATE.csv'
if not DEMOGRAPHICS.exists():
    pd.DataFrame(columns=['platform','campaign_id','date','age_group','gender','city','impressions','clicks','spend_pkr','conversions','data_source']).to_csv(DEMOGRAPHICS,index=False)
print('Optional authorized aggregate template:',DEMOGRAPHICS)
print('Enriched single CSV:',OUTPUT if OUTPUT.exists() else 'not complete yet')

Optional authorized aggregate template: /content/drive/MyDrive/clothing_big_data/authorized_demographic_aggregates_TEMPLATE.csv
Enriched single CSV: not complete yet


## Outputs and interpretation

| File | Meaning |
|---|---|
| `youtube_clothing_ONE_FILE.csv` | Raw collected comments + video metadata (price *mentions* unverified) |
| `clustering_results/youtube_clothing_clustered_ONE_FILE.csv` | Adds TF-IDF topic cluster IDs |
| `clustering_results/video_clusters.csv` | Video engagement/topic distribution clusters |
| `youtube_clothing_enriched_ONE_FILE.csv` | Sentiment predictions + verified product fields (only after full inference) |
| `verified_product_cluster_summary.csv` | Product-level clusters, **only if enough products are genuinely verified** |

**Important:** YouTube does not expose individual commenters' age, gender, or city; authorized aggregate demographics belong in a separate analysis, not attributed to individual comments. Product price mentions are not guaranteed selling prices. Do not interpret video-title candidates as verified SKUs. No million-row completion is implied just by the notebook's capacity.
